# Transpilación de TinyML SIMD-TCN a TFLite Micro INT8 y C++ para ESP32-S3

Este cuaderno toma el modelo Keras entrenado `myotensor_proto_net_tcn_trained.keras` y:
1. Aplica cuantización entera completa **INT8 (Full Integer Quantization)** con dataset representativo.
2. Valida la integridad del archivo `.tflite` y sus operadores (sin control flow ops, $< 30$ KB).
3. Genera automáticamente los archivos C++ directamente en [firmware/Classifier/](file:///home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/):
   - `NN_model.h` (Cabecera)
   - `NN_model.cpp` (Arreglo con `alignas(16)` para extensiones vectoriales SIMD de 128 bits en ESP32-S3)
   - `cnn_scaler_params.h` (Parámetros de normalización Z-score).

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Forzar CPU para conversiones deterministas
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
import tensorflow as tf
import joblib
from pathlib import Path

def load_env_variables():
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()
models_dir = os.environ["MODELS_DL_PROTO"]
base_path = os.environ["PROCESSED_TENSOR_PROTO"]
classifier_root = Path(os.environ["PROJECT_ROOT"]) / "firmware" / "Classifier" 

I0000 00:00:1787778142.971328   48958 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787778143.037615   48958 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787778144.607626   48958 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


In [2]:
# 📥 Cargar modelo entrenado TCN y datos para calibración de cuantización
model_path = os.path.join(models_dir, "myotensor_proto_net_tcn_trained.keras")
print(f"📥 Cargando modelo Keras desde: {model_path}")
model = tf.keras.models.load_model(model_path)
model.summary()

train_data_path = os.path.join(base_path, "X_train.npy")
print(f"📖 Cargando X_train desde: {train_data_path}")
X_train = np.load(train_data_path)
if X_train.ndim == 2:
    X_train = np.expand_dims(X_train, -1)

# Generador de dataset representativo para cuantización INT8
def representative_dataset():
    np.random.seed(42)
    indices = np.random.choice(X_train.shape[0], size=min(300, X_train.shape[0]), replace=False)
    for idx in indices:
        sample = X_train[idx].astype(np.float32)
        yield [np.expand_dims(sample, axis=0)]

print("✅ Generador representative_dataset configurado.")

📥 Cargando modelo Keras desde: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/dl/myotensor_proto_net_tcn_trained.keras


E0000 00:00:1787778145.181981   48958 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1787778145.182031   48958 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES=""
I0000 00:00:1787778145.182046   48958 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to an empty string - this hides all GPUs from CUDA
I0000 00:00:1787778145.182061   48958 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1787778145.182063   48958 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: cbeLOQdebian
I0000 00:00:1787778145.182067   48958 cuda_diagnostics.cc:183] hostname: cbeLOQdebian
I0000 00:00:1787778145.182147   48958 cuda_diagnostics.cc:190] libcuda reported version is: 550.163.1
I0000 00:00:1787778145.182172   48958 cuda_diagnostics.

Model: "SIMD_TCN_MyoTensor"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_semg (InputLayer)     [(None, 200, 1)]             0         []                            
                                                                                                  
 entry_conv (Conv1D)         (None, 100, 16)              64        ['input_semg[0][0]']          
                                                                                                  
 tcn_conv1d_a_1_d1 (Conv1D)  (None, 100, 16)              784       ['entry_conv[0][0]']          
                                                                                                  
 tcn_conv1d_b_1_d1 (Conv1D)  (None, 100, 16)              784       ['tcn_conv1d_a_1_d1[0][0]']   
                                                                                 

In [3]:
# ⚙️ Configurar conversor TFLite para cuantización INT8 completa
print("⚙️ Convirtiendo modelo Keras TCN a TFLite INT8...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
print(f"✅ Conversión INT8 exitosa! Tamaño: {len(tflite_model):,} bytes ({len(tflite_model)/1024:.2f} KB)")

tflite_save_path = os.path.join(models_dir, "myotensor_proto_net_tcn.tflite")
with open(tflite_save_path, "wb") as f:
    f.write(tflite_model)
print(f"💾 Archivo TFLite guardado en: {tflite_save_path}")

⚙️ Convirtiendo modelo Keras TCN a TFLite INT8...
INFO:tensorflow:Assets written to: /tmp/tmpxcku6w37/assets


INFO:tensorflow:Assets written to: /tmp/tmpxcku6w37/assets
/home/cbe/miniconda3/envs/tesis_env/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1787778148.377155   48958 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1787778148.377182   48958 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1787778148.377611   48958 reader.cc:83] Reading SavedModel from: /tmp/tmpxcku6w37
I0000 00:00:1787778148.380030   48958 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1787778148.380046   48958 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpxcku6w37
I0000 00:00:1787778148.393255   48958 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1787778148.396495   48958 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:17877781

✅ Conversión INT8 exitosa! Tamaño: 25,472 bytes (24.88 KB)
💾 Archivo TFLite guardado en: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/dl/myotensor_proto_net_tcn.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1787778149.070839   48958 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.


In [4]:
# 🔍 Validar modelo cuantizado con el Intérprete TFLite
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
print(f"📥 Input TFLite: shape={input_details['shape']}, dtype={input_details['dtype']}, scale={input_details['quantization'][0]}, zero_point={input_details['quantization'][1]}")
print(f"📤 Output TFLite: shape={output_details['shape']}, dtype={output_details['dtype']}")

# Prueba de inferencia rápida
test_sample = (X_train[0:1] / input_details['quantization'][0] + input_details['quantization'][1]).astype(np.int8)
interpreter.set_tensor(input_details['index'], test_sample)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details['index'])
print(f"🎯 Inferencia de prueba exitosa! Salida cuantizada: {output_data}")

📥 Input TFLite: shape=[  1 200   1], dtype=<class 'numpy.int8'>, scale=0.094448022544384, zero_point=30
📤 Output TFLite: shape=[1 4], dtype=<class 'numpy.int8'>
🎯 Inferencia de prueba exitosa! Salida cuantizada: [[ 119 -124 -126 -126]]


/home/cbe/miniconda3/envs/tesis_env/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
# 📝 Escribir archivos C++ (Header y Source) directamente en firmware/Classifier
header_path = classifier_root / "include" / "NN_model.h"
source_path = classifier_root / "src" / "NN_model.cpp"
config_path = classifier_root / "include" / "model_config.h"

# 1. Escribir Header (.h)
header_content = """#pragma once
// Declaración externa del modelo cuantizado TFLite Micro (TinyML-TCN)
extern const unsigned char g_model_data[];
extern const int g_model_data_len;
"""
with open(header_path, "w", encoding="utf-8") as f:
    f.write(header_content)
print(f"✅ Archivo de cabecera creado: {header_path}")

# 2. Formatear bytes a hexadecimal en C++
c_hex_array = [f"0x{val:02X}" for val in tflite_model]
formatted_array = ""
for i in range(0, len(c_hex_array), 12):
    formatted_array += "  " + ", ".join(c_hex_array[i:i+12]) + ",\n"
formatted_array = formatted_array.rstrip(",\n") + "\n"

# 3. Escribir Source (.cpp) con alignas(16) para SIMD de 128 bits en ESP32-S3
source_content = f"""#include \"NN_model.h\"

// Alineación requerida por TensorFlow Lite Micro y SIMD de 128 bits en ESP32-S3
alignas(16) const unsigned char g_model_data[] = {{
{formatted_array}}};

const int g_model_data_len = {len(tflite_model)};
"""
with open(source_path, "w", encoding="utf-8") as f:
    f.write(source_content)
print(f"✅ Archivo de definición creado: {source_path}")

# 4. Actualizar signal_mean y signal_std en model_config.h
std_scaler_file = Path(base_path) / "std_scaler.bin"
if std_scaler_file.exists():
    scaler = joblib.load(std_scaler_file)
    signal_mean = float(scaler.mean_[0])
    signal_std = float(scaler.scale_[0])
else:
    signal_mean = 0.0
    signal_std = 1.0

print(f"✅ Parámetros de señal para CNN/TCN: mean={signal_mean:.6f}, std={signal_std:.6f}")

✅ Archivo de cabecera creado: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/include/NN_model.h
✅ Archivo de definición creado: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/src/NN_model.cpp
✅ Parámetros de escala generados: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/include/cnn_scaler_params.h (mean=-0.046954, std=0.276071)
